# schema

> the shape an extraction fills, and the call that fills it

In [ ]:
#| default_exp schema

Ten built-in shapes, or one you write on the spot as `'vendor:str, total:float'`. `structured`
asks a model for the shape and falls back to a JSON reply when the constrained call fails.

In [ ]:
#| export
import json, re, warnings
from dataclasses import dataclass, fields, is_dataclass, make_dataclass, asdict
from typing import get_origin
from fastcore.all import AttrDict, L

In [ ]:
#| export
@dataclass
class Invoice:
    """An invoice, purchase order or quotation.

    items: one entry per line, each {description, qty, unit_price, amount}.
    Amounts are bare numbers with the currency in `currency`; dates are ISO (2024-03-01).
    """
    number:str = ''
    date:str = ''
    due_date:str = ''
    vendor:str = ''
    vendor_tax_id:str = ''
    bill_to:str = ''
    ship_to:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    payment_terms:str = ''
    items:list[dict] = None

In [ ]:
#| export
@dataclass
class Receipt:
    """A receipt for a completed payment.

    items: one entry per line, each {description, qty, amount}. `paid_with` is the tender or the
    last digits of the card. Dates are ISO (2024-03-01)."""
    merchant:str = ''
    date:str = ''
    transaction_id:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    paid_with:str = ''
    items:list[dict] = None

In [ ]:
#| export
@dataclass
class Catalogue:
    """A product catalogue, price list or listing page.

    products: one entry per product, each {sku, name, category, price, currency, availability, url}.
    Prices are bare numbers."""
    name:str = ''
    vendor:str = ''
    currency:str = ''
    updated:str = ''
    n_products:int = 0
    products:list[dict] = None

In [ ]:
#| export
@dataclass
class Contract:
    """An agreement between parties.

    parties: the legal names. obligations: what each side must do. Dates are ISO (2024-03-01)."""
    title:str = ''
    parties:list[str] = None
    effective_date:str = ''
    end_date:str = ''
    term:str = ''
    value:str = ''
    governing_law:str = ''
    termination:str = ''
    obligations:list[str] = None

In [ ]:
#| export
@dataclass
class Resume:
    """One person's CV.

    experience: one entry per role, each {employer, title, start, end, summary}.
    education: one entry per qualification, each {institution, qualification, year}."""
    name:str = ''
    email:str = ''
    phone:str = ''
    location:str = ''
    headline:str = ''
    years_experience:float = 0
    skills:list[str] = None
    experience:list[dict] = None
    education:list[dict] = None

In [ ]:
#| export
@dataclass
class Paper:
    """An academic paper. authors: names in order. findings: the claims the paper actually makes."""
    title:str = ''
    authors:list[str] = None
    venue:str = ''
    year:str = ''
    doi:str = ''
    abstract:str = ''
    method:str = ''
    findings:list[str] = None
    limitations:list[str] = None

In [ ]:
#| export
@dataclass
class MeetingNotes:
    """Notes from one meeting. actions: one entry per commitment, each {what, owner, due}."""
    title:str = ''
    date:str = ''
    attendees:list[str] = None
    topics:list[str] = None
    decisions:list[str] = None
    actions:list[dict] = None

In [ ]:
#| export
@dataclass
class Summary:
    """What one document says, when no more specific shape fits.

    entities: the organisations, people and places it names. dates: ISO where the document allows."""
    title:str = ''
    doctype:str = ''
    about:str = ''
    key_points:list[str] = None
    entities:list[str] = None
    dates:list[str] = None
    numbers:list[str] = None
    open_questions:list[str] = None

In [ ]:
#| export
# PO and quotation share the Invoice schema
SCHEMAS = dict(invoice=Invoice, purchase_order=Invoice, quote=Invoice, receipt=Receipt,
               catalogue=Catalogue, contract=Contract, resume=Resume, paper=Paper,
               meeting_notes=MeetingNotes, other=Summary)

FIELD_TYPES = {'str':str, 'text':str, 'int':int, 'float':float, 'number':float, 'num':float, 'bool':bool,
               'list':list, 'dict':dict, 'strs':list[str], 'dicts':list[dict],
               'list[str]':list[str], 'list[dict]':list[dict]}
_DFLT = {str: '', int: 0, float: 0.0, bool: False}

In [ ]:
#| export
def dyn_schema(spec,                    # 'vendor:str, total:float, items:dicts', or {'vendor':'str'}, or ['vendor']
               name:str='Extracted',    # class name, which the model sees
               doc:str=None,            # class docstring, which the model also sees
) -> type:
    'A dataclass built at runtime from a field spec: the shape of an answer, named in one string.'
    if isinstance(spec, str): spec = [p for p in re.split(r'[,\n]', spec) if p.strip()]
    if isinstance(spec, dict): spec = [f'{k}:{v}' for k, v in spec.items()]
    flds = []
    for p in spec:
        nm, _, ty = (p if isinstance(p, str) else ':'.join(p)).partition(':')
        t = FIELD_TYPES.get(ty.strip().lower() or 'str', str)
        flds.append((re.sub(r'\W', '_', nm.strip()), t, _DFLT.get(t, None)))
    if not flds: raise ValueError(f'no fields in schema spec {spec!r}')
    return make_dataclass(re.sub(r'\W', '', name) or 'Extracted', flds,
                          namespace=dict(__doc__=doc or 'The fields to pull out of the document.'))

In [ ]:
#| export
def as_schema(spec, name:str='Extracted', doc:str=None) -> type:
    'Whatever names a shape, as a dataclass: a `SCHEMAS` key, a dataclass, or a `dyn_schema` spec.'
    if is_dataclass(spec): return spec
    if isinstance(spec, str) and spec.strip() in SCHEMAS: return SCHEMAS[spec.strip()]
    return dyn_schema(spec, name=name, doc=doc)

In [ ]:
#| export
def _norm(obj, schema) -> dict:
    "A structured reply as a plain dict, with the `None` a list field defaults to turned back into `[]`."
    d = asdict(obj) if is_dataclass(obj) and not isinstance(obj, type) else dict(obj or {})
    lists = {f.name for f in fields(schema) if f.type is list or get_origin(f.type) is list}
    return {k: ([] if v is None and k in lists else v) for k, v in d.items()}

In [ ]:
#| export
def schema_str(schema) -> str:
    "A schema written out for a model to read: its docstring, then `name: type` per field."
    def ty(t):
        if isinstance(t, str): return t
        return str(t).replace('typing.', '') if get_origin(t) else getattr(t, '__name__', str(t))
    return ((schema.__doc__ or '').strip() + '\n\n{\n'
            + ',\n'.join(f'  "{f.name}": {ty(f.type)}' for f in fields(schema)) + '\n}')

In [ ]:
#| export
def _split_reasoning(text:str) -> str:
    "The answer with any reasoning stripped: rishi's `split_think`, plus the *closing*-only tag an MLX prefill leaves."
    from urai import split_think
    text, _ = split_think(text)
    if '</think>' in text: text = text.partition('</think>')[2]
    return text.strip()

In [ ]:
#| export
def _json_reply(ch, prompt:str, schema, sp:str='') -> object:
    'Ask for the shape as a JSON object in prose and rebuild it, with the model left unconstrained.'
    ch.hist = []
    from urai import resp_text, extract_fence
    txt = resp_text(ch(f'{sp}\n\n{prompt}\n\nReply with only a JSON object in a ```json fence, with '
                       f'exactly these keys and types:\n\n{schema_str(schema)}'))
    d = json.loads(extract_fence(_split_reasoning(txt), 'json'))
    nms = {f.name for f in fields(schema)}
    return schema(**{k: v for k, v in d.items() if k in nms})

The default system prompt for an extraction. `structured` takes it as `sp`.


In [ ]:
#| export
EXTRACT_SP = """You pull structured fields out of one document.

Rules:
- Copy what the document states. Do not compute, convert, round or tidy a value.
- A field the document does not state stays empty. An invented number is worse than a blank one.
- Numbers are bare: 1240.50, not "$1,240.50". The currency belongs in its own field.
- Dates are ISO: 2024-03-01.
- A list field takes every entry the document has, in the order it has them."""


In [ ]:
#| export
def structured(ch,                # a `rishi.Chat`, e.g. from `new_chat`
               prompt:str,        # the document and the instruction
               schema,            # the dataclass the reply must fill
               sp:str='',         # system prompt for the extraction
) -> dict:
    'A structured reply as a dict, retried as a plain JSON reply when the constrained call fails.'
    try: return _norm(ch.structured(prompt, schema, sp=sp), schema)
    except Exception as e:
        warnings.warn(f'{type(e).__name__} on a constrained call for {schema.__name__} '
                      f'({str(e)[:150]}); retrying as a JSON reply.')
        return _norm(_json_reply(ch, prompt, schema, sp), schema)

## Tests

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
#| hide
# the prompt is what a model is told, so its shape is the test: a reflow keeps every word and
# still changes the instruction, so a substring check on the words would not catch it
test_eq(EXTRACT_SP, EXTRACT_SP.strip())                    # no leading or trailing blank line
assert 'An invented number is worse' in EXTRACT_SP, EXTRACT_SP
assert EXTRACT_SP.count(chr(10) + chr(10)) >= 1, 'the blank line between the task and the rules is load-bearing'
test_eq(len(EXTRACT_SP.splitlines()[0].split()), len(EXTRACT_SP.splitlines()[0].split()))

In [ ]:
#| hide
# the ten built-in shapes are all dataclasses, and `as_schema` reaches every one by name
test_eq(len(SCHEMAS), 10)
for _n, _s in SCHEMAS.items(): test_eq(is_dataclass(as_schema(_n)), True)
test_eq(as_schema('invoice') is Invoice, True)
test_eq([f.name for f in fields(as_schema('invoice'))][:3], ['number', 'date', 'due_date'])

In [ ]:
#| hide
# a shape made up at the moment of asking, from a string, a dict or a list
_d = as_schema('vendor:str, total:float, items:list')
test_eq([f.name for f in fields(_d)], ['vendor', 'total', 'items'])
test_eq([f.type for f in fields(_d)], [str, float, list])
test_eq([f.name for f in fields(as_schema({'a': 'int', 'b': 'str'}))], ['a', 'b'])
test_eq([f.type for f in fields(as_schema(['a', 'b']))], [str, str])
test_eq([f.name for f in fields(as_schema('total due:float'))], ['total_due'])   # spaces are legal
test_fail(lambda: as_schema(''), contains='no fields')

In [ ]:
#| hide
# `_norm` turns the `None` a list field defaults to back into `[]`, so a caller can always iterate
test_eq(_norm(_d(vendor='Acme', total=1.0, items=None), _d)['items'], [])
test_eq(_norm({'vendor': 'Acme'}, _d), {'vendor': 'Acme'})
# what the model is shown: the docstring, then name and type per field
_txt = schema_str(_d)
assert _txt.startswith('The fields to pull out of the document.'), _txt
for _l in ('"vendor": str', '"total": float', '"items": list'): assert _l in _txt, (_l, _txt)

In [ ]:
#| hide
# every shape has to survive `json.dumps`, or a tool-calling model cannot be handed it:
# a `default_factory` field would break this
import json
from fastcore.funccall import get_schema
for _nm, _s in SCHEMAS.items(): assert json.dumps(get_schema(_s)['input_schema']), _nm


In [ ]:
#| hide
# `structured` prefers the constrained call, and falls back to a JSON reply when it raises
class _Constrained:
    def structured(self, prompt, schema, sp=''): return schema(vendor='Acme', total=9.5, items=['a'])
test_eq(structured(_Constrained(), 'p', _d), {'vendor': 'Acme', 'total': 9.5, 'items': ['a']})

class _JsonOnly:
    hist = []
    def structured(self, *a, **kw): raise RuntimeError('this model has no constrained mode')
    def __call__(self, prompt): return {'content': '```json\n{"vendor": "Acme", "total": 9.5, "junk": 1}\n```'}
import warnings as _w
with _w.catch_warnings():
    _w.simplefilter('ignore')
    _got = structured(_JsonOnly(), 'p', _d)
test_eq(_got, {'vendor': 'Acme', 'total': 9.5, 'items': []})   # `junk` is dropped, `items` filled in

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()